In [ ]:
# ==============================================================================
# CASCATA REAL: INFERÊNCIA FINAL COM PLOT DO SEU COLEGA (COMPACTO)
# ==============================================================================
import ast
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from xgboost import XGBClassifier
from sklearn.svm import NuSVC


from rainfall_acoustic_classification.visualization.plots import plot_confusion_matrix_grid

print("="*60)
print("CALCULANDO A CASCATA REAL E CHAMANDO O PLOT.PY...")
print("="*60)

# 1. LEITURA DOS CSVS
CSV_N1 = f"Resultados_Exp.2.1_Binario_Detalhade_{DATASET}.csv"
CSV_N2 = f"Resultados_Exp.2.2_Intensidade_NuSVC_{DATASET}.csv" 

df_n1 = pd.read_csv(CSV_N1)
df_n2 = pd.read_csv(CSV_N2)

best_n1, best_n2 = df_n1.iloc[0], df_n2.iloc[0]
features_n1 = ast.literal_eval(best_n1['Lista_Features'])
features_n2 = ast.literal_eval(best_n2['Lista_Features'])
algo_n1, algo_n2 = best_n1['Algoritmo'].upper(), best_n2['Algoritmo'].upper()

# 2. INSTANCIAR E TREINAR
def get_model(algo):
    if algo == 'RF': return RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    if algo == 'LR': return LogisticRegression(C=1.0, random_state=42)
    if algo == 'SGD': return SGDClassifier(alpha=0.001, penalty='l2', random_state=42)
    if algo == 'XGB': return XGBClassifier(max_depth=5, learning_rate=0.1, random_state=42)
    if algo == 'NUSVC': return NuSVC(kernel='poly', degree=3, coef0=1.0, nu=0.5, gamma='scale', class_weight='balanced', probability=True, random_state=42)
    return RandomForestClassifier(random_state=42)

modelo_n1, modelo_n2 = get_model(algo_n1), get_model(algo_n2)

y_tr_bin = y_train.apply(lambda x: 'Dry' if x == 'No Rain' else 'Wet')
le_bin = LabelEncoder()
y_tr_bin_enc = pd.Series(le_bin.fit_transform(y_tr_bin), index=y_train.index)
modelo_n1.fit(X_train_selected_metrics[features_n1], y_tr_bin_enc)

mask_tr_wet = y_train != 'No Rain'
le_wet = LabelEncoder()
y_tr_wet_enc = pd.Series(le_wet.fit_transform(y_train[mask_tr_wet]), index=y_train[mask_tr_wet].index)
modelo_n2.fit(X_train_selected_metrics[mask_tr_wet][features_n2], y_tr_wet_enc)

# 3. INFERÊNCIA DA CASCATA
y_pred_n1 = modelo_n1.predict(X_val_selected_metrics[features_n1])
mask_chuva = (y_pred_n1 == list(le_bin.classes_).index('Wet'))

y_pred_final = np.full(len(y_val_enc), le.transform(['No Rain'])[0])

if any(mask_chuva):
    X_wet = X_val_selected_metrics.iloc[mask_chuva][features_n2]
    y_pred_n2 = modelo_n2.predict(X_wet)
    y_pred_final[mask_chuva] = le.transform(le_wet.inverse_transform(y_pred_n2))

print("\nOFFICIAL CASCADE CLASSIFICATION REPORT:")
print(classification_report(y_val_enc, y_pred_final, target_names=le.classes_))

# 4. PREPARAR DADOS E CHAMAR O PLOT DO COLEGA
cm_raw = confusion_matrix(y_val_enc, y_pred_final)
# Multiplicamos por 100 para a matriz de porcentagem
cm_norm = confusion_matrix(y_val_enc, y_pred_final, normalize='true') * 100

titulo = f'Cascade: N1({algo_n1}) + N2({algo_n2})'

# Chama a função nativa do seu projeto!
plot_confusion_matrix_grid(
    cm_raw=cm_raw,
    cm_norm=cm_norm,
    classes=list(le.classes_),
    title=titulo
)

In [ ]:
# ==============================================================================
# 🏁 CASCATA REAL: INFERÊNCIA FINAL E PLOTAGEM COMPACTA
# ==============================================================================
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from xgboost import XGBClassifier
from sklearn.svm import NuSVC

print("="*60)
print("🚀 EXECUTANDO A CASCATA REAL E GERANDO A MATRIZ COMPACTA")
print("="*60)

# ---------------------------------------------------------
# 1. LEITURA DOS CAMPEÕES
# ---------------------------------------------------------
# ATENÇÃO: Confirme se os nomes dos seus CSVs estão exatamente assim:
CSV_N1 = f"Resultados_Exp.1_Binario_D   ET_{DATASET}.csv"
CSV_N2 = f"Resultados_Nivel2_Intensidade_NuSVC_{DATASET}.csv" 

df_n1 = pd.read_csv(CSV_N1)
df_n2 = pd.read_csv(CSV_N2)

# Pega o melhor modelo de cada nível (Linha 0)
best_n1, best_n2 = df_n1.iloc[0], df_n2.iloc[0]

features_n1 = ast.literal_eval(best_n1['Lista_Features'])
features_n2 = ast.literal_eval(best_n2['Lista_Features'])
algo_n1, algo_n2 = best_n1['Algoritmo'].upper(), best_n2['Algoritmo'].upper()

print(f"🥇 Campeão Nível 1 ({algo_n1}): {len(features_n1)} features")
print(f"🥇 Campeão Nível 2 ({algo_n2}): {len(features_n2)} features")

# ---------------------------------------------------------
# 2. INSTANCIAÇÃO DOS MODELOS
# ---------------------------------------------------------
def get_model(algo):
    if algo == 'RF': return RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    if algo == 'LR': return LogisticRegression(C=1.0, random_state=42)
    if algo == 'SGD': return SGDClassifier(alpha=0.001, penalty='l2', random_state=42)
    if algo == 'XGB': return XGBClassifier(max_depth=5, learning_rate=0.1, random_state=42)
    if algo == 'NUSVC': return NuSVC(kernel='poly', degree=3, coef0=1.0, nu=0.5, gamma='scale', class_weight='balanced', probability=True, random_state=42)
    return RandomForestClassifier(random_state=42)

modelo_n1, modelo_n2 = get_model(algo_n1), get_model(algo_n2)

# ---------------------------------------------------------
# 3. TREINO RÁPIDO
# ---------------------------------------------------------
# Treino Nível 1
y_tr_bin = y_train.apply(lambda x: 'Dry' if x == 'No Rain' else 'Wet')
le_bin = LabelEncoder()
y_tr_bin_enc = pd.Series(le_bin.fit_transform(y_tr_bin), index=y_train.index)
modelo_n1.fit(X_train_selected_metrics[features_n1], y_tr_bin_enc)

# Treino Nível 2
mask_tr_wet = y_train != 'No Rain'
le_wet = LabelEncoder()
y_tr_wet_enc = pd.Series(le_wet.fit_transform(y_train[mask_tr_wet]), index=y_train[mask_tr_wet].index)
modelo_n2.fit(X_train_selected_metrics[mask_tr_wet][features_n2], y_tr_wet_enc)

# ---------------------------------------------------------
# 4. INFERÊNCIA EM CASCATA
# ---------------------------------------------------------
# O Nível 1 filtra os dados
y_pred_n1 = modelo_n1.predict(X_val_selected_metrics[features_n1])
idx_wet = list(le_bin.classes_).index('Wet')
mask_chuva = (y_pred_n1 == idx_wet)

# Começa assumindo que não tem chuva
label_no_rain = le.transform(['No Rain'])[0]
y_pred_final = np.full(len(y_val_enc), label_no_rain)

# O Nível 2 classifica apenas o que o Nível 1 disse ser chuva
if any(mask_chuva):
    X_wet = X_val_selected_metrics.iloc[mask_chuva][features_n2]
    y_pred_n2 = modelo_n2.predict(X_wet)
    nomes_n2 = le_wet.inverse_transform(y_pred_n2)
    y_pred_final[mask_chuva] = le.transform(nomes_n2)

# ---------------------------------------------------------
# 5. RELATÓRIO E PLOTAGEM COMPACTA (Pronta para a Tese)
# ---------------------------------------------------------
print("\n📝 RELATÓRIO OFICIAL DA CASCATA INTEGRADA:")
print(classification_report(y_val_enc, y_pred_final, target_names=le.classes_))

cm = confusion_matrix(y_val_enc, y_pred_final)

# Cria a figura bem pequena e compacta
plt.figure(figsize=(5, 5))

# Plota com números grandes e sem barra lateral
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, square=True, 
            annot_kws={"size": 14, "weight": "bold"}, 
            xticklabels=le.classes_, yticklabels=le.classes_)

# Ajuste visual das Labels
plt.xticks(fontsize=12, rotation=45, ha='right')
plt.yticks(fontsize=12, rotation=0)
plt.ylabel('Rótulo Real', fontsize=13, weight='bold')
plt.xlabel('Predição da Cascata', fontsize=13, weight='bold')
plt.title(f'Cascata Real: {algo_n1} ➔ {algo_n2}', fontsize=14, pad=15)

# Corta as bordas desnecessárias
plt.tight_layout(pad=0.5)

# Opcional: Para guardar a imagem compacta em alta resolução, descomente abaixo:
# plt.savefig(f"Matriz_Cascata_Compacta_{DATASET}.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# ==============================================================================
# 🌪️ NÍVEL 2: CLASSIFICADOR DE INTENSIDADE (4 CLASSES) - NuSVC Bare-Metal
# ==============================================================================
import itertools
import time
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.svm import NuSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import average_precision_score
from rainfall_acoustic_classification.modeling import ValidationConfig, ModelEvaluator

print("="*60)
print("🔍 INICIANDO BUSCA DE INTENSIDADE OTIMIZADA - NuSVC Poly Grau 3")
print("="*60)

MIN_FEATURES = 6
MAX_FEATURES = len(top_10_features)
val_config = ValidationConfig(average_method='macro', return_report_dict=True)

# Filtro WET
mask_tr_wet = y_train != 'No Rain'
mask_va_wet = y_val != 'No Rain'
X_tr_wet = X_train_selected_metrics[mask_tr_wet]
y_tr_wet = y_train[mask_tr_wet]
X_va_wet = X_val_selected_metrics[mask_va_wet]
y_va_wet = y_val[mask_va_wet]

le_wet = LabelEncoder()
y_tr_wet_enc = pd.Series(le_wet.fit_transform(y_tr_wet), index=y_tr_wet.index)
y_va_wet_enc = pd.Series(le_wet.transform(y_va_wet), index=y_va_wet.index)
classes_nomes = le_wet.classes_

comb_wet_results = []
contador = 0
total_combinations_pruned = sum([len(list(itertools.combinations(top_10_features, r))) for r in range(MIN_FEATURES, MAX_FEATURES + 1)])

for r in range(MIN_FEATURES, MAX_FEATURES + 1):
    for subset in itertools.combinations(top_10_features, r):
        contador += 1
        subset_list = list(subset)
        
        try:
            clf_master = NuSVC(
                kernel='poly', degree=3, coef0=1.0, nu=0.5, 
                gamma='scale', class_weight='balanced', 
                probability=True, random_state=42
            )
            clf_master.fit(X_tr_wet[subset_list], y_tr_wet_enc)
            
            metrics = ModelEvaluator.evaluate(model=clf_master, X_test=X_va_wet[subset_list], y_test=y_va_wet_enc, config=val_config)
            report = metrics.get('classification_report', {})
            y_proba = metrics.get('y_proba')
            
            row = {
                'Algoritmo': 'NUSVC',
                'F1_Macro': metrics.get('f1_macro', 0.0),
                'Precision_Macro': report.get('macro avg', {}).get('precision', 0.0),
                'Recall_Macro': report.get('macro avg', {}).get('recall', 0.0),
                'Accuracy': report.get('accuracy', 0.0),
                'PR_AUC_Macro': metrics.get('pr_auc_macro', 0.0),
                'Hiperparametros': "kernel='poly', degree=3, nu=0.5",
                'Lista_Features': subset_list
            }

            for idx, nome_classe in enumerate(classes_nomes):
                class_metrics = report.get(str(idx), report.get(nome_classe, {}))
                row[f'{nome_classe}_F1'] = class_metrics.get('f1-score', 0.0)
                row[f'{nome_classe}_Precision'] = class_metrics.get('precision', 0.0)
                row[f'{nome_classe}_Recall'] = class_metrics.get('recall', 0.0)
                
                if y_proba is not None and len(y_proba.shape) == 2 and y_proba.shape[1] > idx:
                    y_true_binary = (y_va_wet_enc == idx).astype(int)
                    row[f'{nome_classe}_PR_AUC'] = average_precision_score(y_true_binary, y_proba[:, idx])
                else:
                    row[f'{nome_classe}_PR_AUC'] = 0.0

            comb_wet_results.append(row)
            
        except Exception as e: 
            continue
            
        if contador % 50 == 0:
            melhor = max([res['F1_Macro'] for res in comb_wet_results]) if comb_wet_results else 0.0
            print(f"   -> Progresso: {contador}/{total_combinations_pruned}... Melhor F1: {melhor:.4f}")

if len(comb_wet_results) > 0:
    df_wet = pd.DataFrame(comb_wet_results).sort_values(by='F1_Macro', ascending=False).reset_index(drop=True)
    csv_name_wet = f"Resultados_Nivel2_Intensidade_NuSVC_{DATASET}.csv"
    df_wet.to_csv(csv_name_wet, index=False)
    print(f"\nCSV NuSVC Nível 2 gerado: {csv_name_wet}")
    display(df_wet.head(2))

In [ ]:
# ==============================================================================
# 🚰 NÍVEL 1: DETECTOR DE CHUVA (DRY vs. WET) - NuSVC Bare-Metal (CSV Detalhado)
# ==============================================================================
import itertools
import time
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.svm import NuSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import average_precision_score
from rainfall_acoustic_classification.modeling import ValidationConfig, ModelEvaluator

print("="*60)
print("🌧️ INICIANDO BUSCA BINÁRIA OTIMIZADA - NuSVC Poly Grau 3")
print("="*60)

MIN_FEATURES = 6
MAX_FEATURES = len(top_10_features)
val_config = ValidationConfig(average_method='macro', return_report_dict=True)

# Filtro e Codificação Segura
y_train_bin = y_train.apply(lambda x: 'Dry' if x == 'No Rain' else 'Wet')
y_val_bin   = y_val.apply(lambda x: 'Dry' if x == 'No Rain' else 'Wet')

le_bin = LabelEncoder()
y_tr_bin_enc = pd.Series(le_bin.fit_transform(y_train_bin), index=y_train.index)
y_va_bin_enc = pd.Series(le_bin.transform(y_val_bin), index=y_val.index)
classes_nomes_bin = le_bin.classes_

comb_bin_results = []
contador = 0
total_combinations_pruned = sum([len(list(itertools.combinations(top_10_features, r))) for r in range(MIN_FEATURES, MAX_FEATURES + 1)])

for r in range(MIN_FEATURES, MAX_FEATURES + 1):
    for subset in itertools.combinations(top_10_features, r):
        contador += 1
        subset_list = list(subset)
        
        try:
            # Bypass: Instanciação e Treino Direto
            clf_master = NuSVC(
                kernel='poly', degree=3, coef0=1.0, nu=0.5, 
                gamma='scale', class_weight='balanced', 
                probability=True, random_state=42
            )
            clf_master.fit(X_train_selected_metrics[subset_list], y_tr_bin_enc)
            
            # Avaliação
            metrics = ModelEvaluator.evaluate(model=clf_master, X_test=X_val_selected_metrics[subset_list], y_test=y_va_bin_enc, config=val_config)
            report = metrics.get('classification_report', {})
            y_proba = metrics.get('y_proba')
            
            row = {
                'Algoritmo': 'NUSVC',
                'F1_Macro': metrics.get('f1_macro', 0.0),
                'Precision_Macro': report.get('macro avg', {}).get('precision', 0.0),
                'Recall_Macro': report.get('macro avg', {}).get('recall', 0.0),
                'Accuracy': report.get('accuracy', 0.0),
                'PR_AUC_Macro': metrics.get('pr_auc_macro', 0.0),
                'Hiperparametros': "kernel='poly', degree=3, nu=0.5",
                'Lista_Features': subset_list
            }

            # Extração por Classe
            for idx, nome_classe in enumerate(classes_nomes_bin):
                class_metrics = report.get(str(idx), report.get(nome_classe, {}))
                row[f'{nome_classe}_F1'] = class_metrics.get('f1-score', 0.0)
                row[f'{nome_classe}_Precision'] = class_metrics.get('precision', 0.0)
                row[f'{nome_classe}_Recall'] = class_metrics.get('recall', 0.0)
                
                if y_proba is not None and len(y_proba.shape) == 2 and y_proba.shape[1] > idx:
                    y_true_binary = (y_va_bin_enc == idx).astype(int)
                    row[f'{nome_classe}_PR_AUC'] = average_precision_score(y_true_binary, y_proba[:, idx])
                else:
                    row[f'{nome_classe}_PR_AUC'] = 0.0

            comb_bin_results.append(row)
            
        except Exception as e: 
            continue
            
        if contador % 50 == 0:
            melhor_atual = max([res['F1_Macro'] for res in comb_bin_results]) if comb_bin_results else 0.0
            print(f"   -> Progresso: {contador}/{total_combinations_pruned}... Melhor F1: {melhor_atual:.4f}")

if len(comb_bin_results) > 0:
    df_bin = pd.DataFrame(comb_bin_results).sort_values(by='F1_Macro', ascending=False).reset_index(drop=True)
    csv_name_bin = f"Resultados_Nivel1_Binario_NuSVC_{DATASET}.csv"
    df_bin.to_csv(csv_name_bin, index=False)
    print(f"\n✅ CSV NuSVC Nível 1 gerado: {csv_name_bin}")
    display(df_bin.head(2))

In [ ]:
# ==============================================================================
# 🌪️ NÍVEL 2: CLASSIFICADOR DE INTENSIDADE (4 CLASSES) - LR, SGD e XGBOOST
# ==============================================================================
import itertools
import time
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import average_precision_score
from rainfall_acoustic_classification.modeling import ValidationConfig, ClassifierConfig, TuningConfig, ClassifierFactory, ModelOptimizer, ModelEvaluator

print("="*60)
print("🔍 INICIANDO BUSCA DE INTENSIDADE (NÍVEL 2) - LR, SGD e XGBoost")
print("="*60)

# 0. Setup
val_config = ValidationConfig(average_method='macro', return_report_dict=True)

# 1. Filtro WET e Codificação (Remove o 'No Rain')
mask_tr_wet = y_train != 'No Rain'
mask_va_wet = y_val != 'No Rain'
X_tr_wet = X_train_selected_metrics[mask_tr_wet]
y_tr_wet = y_train[mask_tr_wet]
X_va_wet = X_val_selected_metrics[mask_va_wet]
y_va_wet = y_val[mask_va_wet]

le_wet = LabelEncoder()
y_tr_wet_enc = pd.Series(le_wet.fit_transform(y_tr_wet), index=y_tr_wet.index)
y_va_wet_enc = pd.Series(le_wet.transform(y_va_wet), index=y_va_wet.index)
classes_nomes = le_wet.classes_ # ['Heavy', 'Light', 'Moderate', 'Violent']

# 2. Configurações dos Modelos para o Nível 2
modelos_nivel2 = {
    'lr': TuningConfig(param_grid={'C': [1.0]}, scoring_metric='f1_macro', n_jobs=-1),
    'sgd': TuningConfig(param_grid={'alpha': [0.001], 'penalty': ['l2']}, scoring_metric='f1_macro', n_jobs=-1),
    'xgb': TuningConfig(param_grid={'max_depth': [5], 'learning_rate': [0.1]}, scoring_metric='f1_macro', n_jobs=-1)
}

# 3. Loop de Busca Exaustiva por Modelo
for model_name, tuning_config in modelos_nivel2.items():
    print(f"\n🚀 Processando {model_name.upper()} nas 4 intensidades de chuva...")
    config_model = ClassifierConfig(model_type=model_name, random_state=42)
    
    comb_wet_results = []
    contador = 0
    
    for r in range(1, 11):
        for subset in itertools.combinations(top_10_features, r):
            contador += 1
            subset_list = list(subset)
            
            try:
                campeao = ModelOptimizer.optimize(
                    estimator=ClassifierFactory.build(config_model),
                    X_train=X_tr_wet[subset_list], y_train=y_tr_wet_enc,
                    X_val=X_va_wet[subset_list], y_val=y_va_wet_enc, config=tuning_config
                )
                
                metrics = ModelEvaluator.evaluate(model=campeao, X_test=X_va_wet[subset_list], y_test=y_va_wet_enc, config=val_config)
                
                report = metrics.get('classification_report', {})
                y_proba = metrics.get('y_proba')
                
                row = {
                    'Algoritmo': model_name.upper(),
                    'F1_Macro': metrics.get('f1_macro', 0.0),
                    'Precision_Macro': report.get('macro avg', {}).get('precision', 0.0),
                    'Recall_Macro': report.get('macro avg', {}).get('recall', 0.0),
                    'Accuracy': report.get('accuracy', 0.0),
                    'PR_AUC_Macro': metrics.get('pr_auc_macro', 0.0),
                    'Hiperparametros': str(tuning_config.param_grid),
                    'Lista_Features': subset_list
                }

                # --- Extração Dinâmica por Classe (Com a Correção Numérica) ---
                for idx, nome_classe in enumerate(classes_nomes):
                    # Procura a chave numérica (ex: '0'), se não achar procura o nome
                    class_metrics = report.get(str(idx), report.get(nome_classe, {}))
                    
                    row[f'{nome_classe}_F1'] = class_metrics.get('f1-score', 0.0)
                    row[f'{nome_classe}_Precision'] = class_metrics.get('precision', 0.0)
                    row[f'{nome_classe}_Recall'] = class_metrics.get('recall', 0.0)
                    
                    # Cálculo do PR-AUC
                    if y_proba is not None and len(y_proba.shape) == 2 and y_proba.shape[1] > idx:
                        y_true_binary = (y_va_wet_enc == idx).astype(int)
                        row[f'{nome_classe}_PR_AUC'] = average_precision_score(y_true_binary, y_proba[:, idx])
                    else:
                        row[f'{nome_classe}_PR_AUC'] = 0.0

                comb_wet_results.append(row)
                
            except Exception as e: 
                # Ignora erros pontuais (como XGBoost reclamando de parâmetros) para não parar o loop
                continue 
            
            if contador % 200 == 0:
                melhor = max([res['F1_Macro'] for res in comb_wet_results]) if comb_wet_results else 0.0
                print(f"   -> {model_name.upper()}: {contador}/1023... Melhor F1 Macro: {melhor:.4f}")

    # 4. Salvar e Exibir o CSV específico deste modelo
    if len(comb_wet_results) > 0:
        df_wet = pd.DataFrame(comb_wet_results).sort_values(by='F1_Macro', ascending=False).reset_index(drop=True)
        csv_name_wet = f"Resultados_Nivel2_Intensidade_{model_name.upper()}_Detalhado_{DATASET}.csv"
        df_wet.to_csv(csv_name_wet, index=False)

        print(f"\n✅ Busca do {model_name.upper()} concluída! CSV salvo: {csv_name_wet}")
        display(df_wet.head(2)) # Exibe só o top 2 para não poluir muito a tela
    else:
        print(f"\n⚠️ Erro ao gerar dados para o modelo {model_name.upper()}.")

🔍 INICIANDO BUSCA DE INTENSIDADE (NÍVEL 2) - LR, SGD e XGBoost

🚀 Processando LR nas 4 intensidades de chuva...
14:31:09 - [ClassifierFactory] - INFO - CPU Scaling: Requested -1 -> Allocated 14 cores.
14:31:09 - [ClassifierFactory] - INFO - Building registered model architecture: LR
14:31:09 - [HyperparameterTuner] - INFO - Starting Optimization Engine. Target Metric: f1_macro
Fitting 1 folds for each of 1 candidates, totalling 1 fits
14:31:12 - [HyperparameterTuner] - INFO - Optimization complete. Champion Score (f1_macro) on Val Set: 0.2441
14:31:12 - [HyperparameterTuner] - INFO - Champion Hyperparameters: {'C': 1.0}
14:31:12 - [HyperparameterTuner] - INFO - Refitting the champion model exclusively on the Training Set...
14:31:13 - [ModelEvaluator] - INFO - Executing Rigorous Stress Test (Model Evaluation)...
14:31:13 - [ModelEvaluator] - INFO - Test Results -> F1-Macro: 0.2441 | PR-AUC: 0.272027791572494
14:31:13 - [ClassifierFactory] - INFO - Building registered model architecture

,Algoritmo,F1_Macro,Precision_Macro,Recall_Macro,Accuracy,PR_AUC_Macro,Hiperparametros,Lista_Features,Heavy_F1,Heavy_Precision,...,Light_Recall,Light_PR_AUC,Moderate_F1,Moderate_Precision,Moderate_Recall,Moderate_PR_AUC,Violent_F1,Violent_Precision,Violent_Recall,Violent_PR_AUC
0,LR,0.391969,0.410187,0.417803,0.450460,0.340046,{'C': [1.0]},"[mfcc_6, mfcc_3, wav_detail_lvl3_std, mfcc_std]",0.363636,0.545455,...,0.60303,0.448758,0.455882,0.457227,0.454545,0.416807,0.209790,0.151515,0.340909,0.081578
1,LR,0.381976,0.404426,0.411694,0.444331,0.338129,{'C': [1.0]},"[mfcc_6, mfcc_3, wav_detail_lvl3_std, mfcc_1]",0.333333,0.533333,...,0.60303,0.450276,0.459736,0.459064,0.460411,0.416934,0.194805,0.136364,0.340909,0.080648



🚀 Processando SGD nas 4 intensidades de chuva...
14:37:03 - [ClassifierFactory] - INFO - CPU Scaling: Requested -1 -> Allocated 14 cores.
14:37:03 - [ClassifierFactory] - INFO - Building registered model architecture: SGD
14:37:03 - [HyperparameterTuner] - INFO - Starting Optimization Engine. Target Metric: f1_macro
Fitting 1 folds for each of 1 candidates, totalling 1 fits
14:37:03 - [HyperparameterTuner] - INFO - Optimization complete. Champion Score (f1_macro) on Val Set: 0.1099
14:37:03 - [HyperparameterTuner] - INFO - Champion Hyperparameters: {'alpha': 0.001, 'penalty': 'l2'}
14:37:03 - [HyperparameterTuner] - INFO - Refitting the champion model exclusively on the Training Set...
14:37:03 - [ModelEvaluator] - INFO - Executing Rigorous Stress Test (Model Evaluation)...
14:37:03 - [ModelEvaluator] - INFO - Test Results -> F1-Macro: 0.1099 | PR-AUC: 0.2681233447812601
14:37:03 - [ClassifierFactory] - INFO - Building registered model architecture: SGD
14:37:03 - [HyperparameterTuner

,Algoritmo,F1_Macro,Precision_Macro,Recall_Macro,Accuracy,PR_AUC_Macro,Hiperparametros,Lista_Features,Heavy_F1,Heavy_Precision,...,Light_Recall,Light_PR_AUC,Moderate_F1,Moderate_Precision,Moderate_Recall,Moderate_PR_AUC,Violent_F1,Violent_Precision,Violent_Recall,Violent_PR_AUC
0,SGD,0.382333,0.407446,0.410734,0.439224,0.336687,"{'alpha': [0.001], 'penalty': ['l2']}","[mfcc_6, mfcc_3, rms_mean, mfcc_5]",0.385965,0.570370,...,0.593939,0.438027,0.439628,0.465574,0.416422,0.42395,0.175439,0.11811,0.340909,0.087799
1,SGD,0.381331,0.405983,0.409787,0.438202,0.338140,"{'alpha': [0.001], 'penalty': ['l2']}","[mfcc_6, mfcc_3, mae, mfcc_5]",0.381910,0.567164,...,0.593939,0.438246,0.437596,0.461039,0.416422,0.42527,0.177515,0.12000,0.340909,0.089558



🚀 Processando XGB nas 4 intensidades de chuva...
14:40:14 - [ClassifierFactory] - INFO - CPU Scaling: Requested -1 -> Allocated 14 cores.
14:40:14 - [ClassifierFactory] - INFO - Building registered model architecture: XGB
14:40:14 - [HyperparameterTuner] - INFO - Starting Optimization Engine. Target Metric: f1_macro
Fitting 1 folds for each of 1 candidates, totalling 1 fits
14:40:16 - [HyperparameterTuner] - INFO - Optimization complete. Champion Score (f1_macro) on Val Set: 0.2123
14:40:16 - [HyperparameterTuner] - INFO - Champion Hyperparameters: {'learning_rate': 0.1, 'max_depth': 5}
14:40:16 - [HyperparameterTuner] - INFO - Refitting the champion model exclusively on the Training Set...
14:40:17 - [ModelEvaluator] - INFO - Executing Rigorous Stress Test (Model Evaluation)...
14:40:17 - [ModelEvaluator] - INFO - Test Results -> F1-Macro: 0.2123 | PR-AUC: 0.26063285978835127
14:40:17 - [ClassifierFactory] - INFO - Building registered model architecture: XGB
14:40:17 - [Hyperparamete

,Algoritmo,F1_Macro,Precision_Macro,Recall_Macro,Accuracy,PR_AUC_Macro,Hiperparametros,Lista_Features,Heavy_F1,Heavy_Precision,...,Light_Recall,Light_PR_AUC,Moderate_F1,Moderate_Precision,Moderate_Recall,Moderate_PR_AUC,Violent_F1,Violent_Precision,Violent_Recall,Violent_PR_AUC
0,XGB,0.377755,0.372112,0.402535,0.421859,0.360394,"{'max_depth': [5], 'learning_rate': [0.1]}","[wav_detail_lvl5_std, mfcc_3, mfcc_5]",0.381503,0.388235,...,0.539394,0.536078,0.382911,0.415808,0.354839,0.367865,0.211268,0.153061,0.340909,0.099645
1,XGB,0.377382,0.385329,0.457337,0.407559,0.391455,"{'max_depth': [5], 'learning_rate': [0.1]}","[mfcc_3, mfcc_5]",0.408730,0.429167,...,0.524242,0.476496,0.346084,0.456731,0.278592,0.422103,0.238298,0.146597,0.636364,0.193394


In [ ]:
# ==============================================================================
# 🌪️ NÍVEL 2: CLASSIFICADOR DE INTENSIDADE (4 CLASSES) - CSV DETALHADO
# ==============================================================================
import itertools
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import average_precision_score
from rainfall_acoustic_classification.modeling import ValidationConfig, ClassifierConfig, TuningConfig, ClassifierFactory, ModelOptimizer, ModelEvaluator

print("="*60)
print("🔍 INICIANDO BUSCA DE INTENSIDADE (DETALHE POR CLASSE) PARA CSV...")
print("="*60)

# 0. Setup
val_config = ValidationConfig(average_method='macro', return_report_dict=True)

# 1. Filtro e Codificação
mask_tr_wet = y_train != 'No Rain'
mask_va_wet = y_val != 'No Rain'
X_tr_wet = X_train_selected_metrics[mask_tr_wet]
y_tr_wet = y_train[mask_tr_wet]
X_va_wet = X_val_selected_metrics[mask_va_wet]
y_va_wet = y_val[mask_va_wet]

le_wet = LabelEncoder()
y_tr_wet_enc = pd.Series(le_wet.fit_transform(y_tr_wet), index=y_tr_wet.index)
y_va_wet_enc = pd.Series(le_wet.transform(y_va_wet), index=y_va_wet.index)
classes_nomes = le_wet.classes_ # ['Heavy', 'Light', 'Moderate', 'Violent']

# 2. Configuração RF
rf_config_wet = ClassifierConfig(model_type='rf', random_state=42)
tuning_config_wet = TuningConfig(param_grid={'n_estimators': [100], 'max_depth': [10]}, scoring_metric='f1_macro', n_jobs=-1)

comb_wet_results = []
for r in range(1, 11):
    for subset in itertools.combinations(top_10_features, r):
        subset_list = list(subset)
        try:
            rf_champion = ModelOptimizer.optimize(
                estimator=ClassifierFactory.build(rf_config_wet),
                X_train=X_tr_wet[subset_list], y_train=y_tr_wet_enc,
                X_val=X_va_wet[subset_list], y_val=y_va_wet_enc, config=tuning_config_wet
            )
            metrics = ModelEvaluator.evaluate(model=rf_champion, X_test=X_va_wet[subset_list], y_test=y_va_wet_enc, config=val_config)
            
            report = metrics.get('classification_report', {})
            y_proba = metrics.get('y_proba')
            
            # Dicionário da Linha para o CSV
            row = {
                'Algoritmo': 'RF',
                'F1_Macro': metrics.get('f1_macro', 0.0),
                'Precision_Macro': report.get('macro avg', {}).get('precision', 0.0),
                'Recall_Macro': report.get('macro avg', {}).get('recall', 0.0),
                'Accuracy': report.get('accuracy', 0.0),
                'PR_AUC_Macro': metrics.get('pr_auc_macro', 0.0),
                'Hiperparametros': str(tuning_config_wet.param_grid),
                'Lista_Features': subset_list
            }

            # --- Extração Dinâmica por Classe ---
            for idx, nome_classe in enumerate(classes_nomes):
                # Métricas do Relatório (F1, Precision, Recall)
                class_metrics = report.get(str(idx), report.get(nome_classe, {}))
                row[f'{nome_classe}_F1'] = class_metrics.get('f1-score', 0.0)
                row[f'{nome_classe}_Precision'] = class_metrics.get('precision', 0.0)
                row[f'{nome_classe}_Recall'] = class_metrics.get('recall', 0.0)
                
                # Cálculo do PR-AUC por classe (One-vs-Rest)
                if y_proba is not None:
                    y_true_binary = (y_va_wet_enc == idx).astype(int)
                    row[f'{nome_classe}_PR_AUC'] = average_precision_score(y_true_binary, y_proba[:, idx])
                else:
                    row[f'{nome_classe}_PR_AUC'] = 0.0

            comb_wet_results.append(row)
            
        except Exception as e: 
            print(f"❌ Erro na combinação {subset_list}: {e}")
            break 
            
    if len(comb_wet_results) == 0 and r == 1:
        break

# 3. Salvar e Exibir
if len(comb_wet_results) > 0:
    df_wet = pd.DataFrame(comb_wet_results).sort_values(by='F1_Macro', ascending=False).reset_index(drop=True)
    csv_name_wet = f"Resultados_Nivel2_Intensidade_Detalhado_{DATASET}_RF.csv"
    df_wet.to_csv(csv_name_wet, index=False)

    print(f"\n✅ CSV Detalhado gerado: {csv_name_wet}")
    display(df_wet.head(5))
    melhores_features_wet = df_wet.iloc[0]['Lista_Features']
else:
    print("\n⚠️ Erro ao gerar dados.")

🔍 INICIANDO BUSCA DE INTENSIDADE (DETALHE POR CLASSE) PARA CSV...
14:59:29 - [ClassifierFactory] - INFO - CPU Scaling: Requested -1 -> Allocated 14 cores.
14:59:29 - [ClassifierFactory] - INFO - Building registered model architecture: RF
14:59:29 - [HyperparameterTuner] - INFO - Starting Optimization Engine. Target Metric: f1_macro
Fitting 1 folds for each of 1 candidates, totalling 1 fits
14:59:30 - [HyperparameterTuner] - INFO - Optimization complete. Champion Score (f1_macro) on Val Set: 0.2053
14:59:30 - [HyperparameterTuner] - INFO - Champion Hyperparameters: {'max_depth': 10, 'n_estimators': 100}
14:59:30 - [HyperparameterTuner] - INFO - Refitting the champion model exclusively on the Training Set...
14:59:30 - [ModelEvaluator] - INFO - Executing Rigorous Stress Test (Model Evaluation)...
14:59:30 - [ModelEvaluator] - INFO - Test Results -> F1-Macro: 0.2053 | PR-AUC: 0.26082340021935946
14:59:30 - [ClassifierFactory] - INFO - Building registered model architecture: RF
14:59:30 - 

,Algoritmo,F1_Macro,Precision_Macro,Recall_Macro,Accuracy,PR_AUC_Macro,Hiperparametros,Lista_Features,Heavy_F1,Heavy_Precision,...,Light_Recall,Light_PR_AUC,Moderate_F1,Moderate_Precision,Moderate_Recall,Moderate_PR_AUC,Violent_F1,Violent_Precision,Violent_Recall,Violent_PR_AUC
0,RF,0.388331,0.380636,0.420002,0.427988,0.354566,"{'n_estimators': [100], 'max_depth': [10]}","[wav_detail_lvl5_std, mfcc_3, mfcc_5]",0.362173,0.386266,...,0.557576,0.530192,0.391975,0.413681,0.372434,0.363812,0.248276,0.178218,0.409091,0.133597
1,RF,0.368188,0.369079,0.388746,0.411645,0.340076,"{'n_estimators': [100], 'max_depth': [10]}","[wav_detail_lvl5_std, mfcc_1, mfcc_std]",0.404301,0.467662,...,0.484848,0.474967,0.404798,0.414110,0.395894,0.388813,0.194444,0.140000,0.318182,0.091740
2,RF,0.366836,0.364648,0.389467,0.414709,0.333294,"{'n_estimators': [100], 'max_depth': [10]}","[wav_detail_lvl5_std, mfcc_3, wav_detail_lvl4_...",0.360515,0.415842,...,0.566667,0.521902,0.364458,0.374613,0.354839,0.346127,0.197183,0.142857,0.318182,0.094260
3,RF,0.366747,0.367300,0.385423,0.413687,0.344500,"{'n_estimators': [100], 'max_depth': [10]}","[wav_detail_lvl5_std, mfcc_1]",0.401674,0.448598,...,0.451515,0.475139,0.435556,0.440120,0.431085,0.409807,0.176871,0.126214,0.295455,0.093963
4,RF,0.365295,0.360299,0.396805,0.398366,0.337009,"{'n_estimators': [100], 'max_depth': [10]}","[mfcc_3, mae, mfcc_5]",0.344398,0.380734,...,0.503030,0.514728,0.359649,0.358601,0.360704,0.348996,0.240000,0.169811,0.409091,0.120738


In [ ]:
# ==============================================================================
# 🚰 NÍVEL 1: DETECTOR DE CHUVA (DRY vs. WET) - 4 CLASSIFICADORES (CSV DETALHADO)
# ==============================================================================
import itertools
import time
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import average_precision_score
from rainfall_acoustic_classification.modeling import ValidationConfig, ClassifierConfig, TuningConfig, ClassifierFactory, ModelOptimizer, ModelEvaluator

print("="*60)
print("🌧️ INICIANDO BUSCA BINÁRIA (DETALHE POR CLASSE) PARA CSV...")
print("="*60)

# 0. Setup
val_config = ValidationConfig(average_method='macro', return_report_dict=True)

# 1. Filtro e Codificação
y_train_bin = y_train.apply(lambda x: 'Dry' if x == 'No Rain' else 'Wet')
y_val_bin   = y_val.apply(lambda x: 'Dry' if x == 'No Rain' else 'Wet')

le_bin = LabelEncoder()
y_tr_bin_enc = pd.Series(le_bin.fit_transform(y_train_bin), index=y_train.index)
y_va_bin_enc = pd.Series(le_bin.transform(y_val_bin), index=y_val.index)
classes_nomes_bin = le_bin.classes_ # Será ['Dry', 'Wet']

# 2. Configurações dos 4 Modelos
modelos_binarios = {
    'rf': TuningConfig(param_grid={'n_estimators': [100], 'max_depth': [10]}, scoring_metric='f1_macro', n_jobs=-1),
    #'lr': TuningConfig(param_grid={'C': [1.0]}, scoring_metric='f1_macro', n_jobs=-1),
    #'sgd': TuningConfig(param_grid={'alpha': [0.001], 'penalty': ['l2']}, scoring_metric='f1_macro', n_jobs=-1),
    'xgb': TuningConfig(param_grid={'max_depth': [5], 'learning_rate': [0.1]}, scoring_metric='f1_macro', n_jobs=-1)
}

comb_bin_results = []

for model_name, tuning_config in modelos_binarios.items():
    print(f"\n🚀 Processando {model_name.upper()}...")
    config_bin = ClassifierConfig(model_type=model_name, random_state=42)
    contador = 0
    
    for r in range(1, 11):
        for subset in itertools.combinations(top_10_features, r):
            contador += 1
            subset_list = list(subset)
            try:
                campeao = ModelOptimizer.optimize(
                    estimator=ClassifierFactory.build(config_bin),
                    X_train=X_train_selected_metrics[subset_list], y_train=y_tr_bin_enc,
                    X_val=X_val_selected_metrics[subset_list], y_val=y_va_bin_enc, config=tuning_config
                )
                metrics = ModelEvaluator.evaluate(model=campeao, X_test=X_val_selected_metrics[subset_list], y_test=y_va_bin_enc, config=val_config)
                
                report = metrics.get('classification_report', {})
                y_proba = metrics.get('y_proba')
                
                # Dicionário Base para a linha do CSV
                row = {
                    'Algoritmo': model_name.upper(),
                    'F1_Macro': metrics.get('f1_macro', 0.0),
                    'Precision_Macro': report.get('macro avg', {}).get('precision', 0.0),
                    'Recall_Macro': report.get('macro avg', {}).get('recall', 0.0),
                    'Accuracy': report.get('accuracy', 0.0),
                    'PR_AUC_Macro': metrics.get('pr_auc_macro', 0.0),
                    'Hiperparametros': str(tuning_config.param_grid),
                    'Lista_Features': subset_list
                }

                # --- Extração Dinâmica por Classe (Dry e Wet) ---
                for idx, nome_classe in enumerate(classes_nomes_bin):
                    class_metrics = report.get(str(idx), report.get(nome_classe, {}))
                    row[f'{nome_classe}_F1'] = class_metrics.get('f1-score', 0.0)
                    row[f'{nome_classe}_Precision'] = class_metrics.get('precision', 0.0)
                    row[f'{nome_classe}_Recall'] = class_metrics.get('recall', 0.0)
                    
                    # Cálculo Seguro do PR-AUC
                    if y_proba is not None and len(y_proba.shape) == 2 and y_proba.shape[1] > idx:
                        y_true_binary = (y_va_bin_enc == idx).astype(int)
                        row[f'{nome_classe}_PR_AUC'] = average_precision_score(y_true_binary, y_proba[:, idx])
                    else:
                        row[f'{nome_classe}_PR_AUC'] = 0.0

                comb_bin_results.append(row)
                
            except Exception as e: 
                # Ignora erros silenciosamente para que um modelo não pare a execução dos outros
                continue
                
        if contador % 200 == 0:
            melhor_atual = max([res['F1_Macro'] for res in comb_bin_results if res['Algoritmo'] == model_name.upper()]) if any(res['Algoritmo'] == model_name.upper() for res in comb_bin_results) else 0.0
            print(f"   -> Progresso {model_name.upper()}: {contador}/1023... Melhor F1 Macro: {melhor_atual:.4f}")

# 3. Salvar e Exibir
if len(comb_bin_results) > 0:
    df_bin = pd.DataFrame(comb_bin_results).sort_values(by='F1_Macro', ascending=False).reset_index(drop=True)
    
    # Criamos um único CSV consolidado contendo os 4 modelos
    csv_name_bin = f"Resultados_Exp.1_Binario_{DATASET}.csv"
    df_bin.to_csv(csv_name_bin, index=False)
    
    print(f"\n✅ Busca Binária Concluída! CSV consolidado gerado: {csv_name_bin}")
    display(df_bin.head(5))
    
    melhor_modelo_bin_nome = df_bin.iloc[0]['Algoritmo'].lower()
    melhores_features_bin  = df_bin.iloc[0]['Lista_Features']
    print(f"🥇 Campeão Geral do Nível 1: {melhor_modelo_bin_nome.upper()}")
else:
    print("\n⚠️ A tabela e o CSV não foram gerados devido a erros em todos os modelos.")

🌧️ INICIANDO BUSCA BINÁRIA (DETALHE POR CLASSE) PARA CSV...

🚀 Processando RF...
13:50:48 - [ClassifierFactory] - INFO - CPU Scaling: Requested -1 -> Allocated 14 cores.
13:50:48 - [ClassifierFactory] - INFO - Building registered model architecture: RF
13:50:48 - [HyperparameterTuner] - INFO - Starting Optimization Engine. Target Metric: f1_macro


Fitting 1 folds for each of 1 candidates, totalling 1 fits
13:50:52 - [HyperparameterTuner] - INFO - Optimization complete. Champion Score (f1_macro) on Val Set: 0.8591
13:50:52 - [HyperparameterTuner] - INFO - Champion Hyperparameters: {'max_depth': 10, 'n_estimators': 100}
13:50:52 - [HyperparameterTuner] - INFO - Refitting the champion model exclusively on the Training Set...
13:50:52 - [ModelEvaluator] - INFO - Executing Rigorous Stress Test (Model Evaluation)...
13:50:52 - [ModelEvaluator] - INFO - Test Results -> F1-Macro: 0.8591 | PR-AUC: 0.9362689064716758
13:50:52 - [ClassifierFactory] - INFO - Building registered model architecture: RF
13:50:52 - [HyperparameterTuner] - INFO - Starting Optimization Engine. Target Metric: f1_macro
Fitting 1 folds for each of 1 candidates, totalling 1 fits
13:50:55 - [HyperparameterTuner] - INFO - Optimization complete. Champion Score (f1_macro) on Val Set: 0.6907
13:50:55 - [HyperparameterTuner] - INFO - Champion Hyperparameters: {'max_depth':

,Algoritmo,F1_Macro,Precision_Macro,Recall_Macro,Accuracy,PR_AUC_Macro,Hiperparametros,Lista_Features,Dry_F1,Dry_Precision,Dry_Recall,Dry_PR_AUC,Wet_F1,Wet_Precision,Wet_Recall,Wet_PR_AUC
0,RF,0.913452,0.917004,0.913415,0.913662,0.966968,"{'n_estimators': [100], 'max_depth': [10]}","[mfcc_3, wav_detail_lvl3_std, wav_detail_lvl4_...",0.917715,0.881041,0.957576,0.954045,0.909188,0.952968,0.869254,0.966968
1,RF,0.912929,0.916725,0.912899,0.913154,0.968291,"{'n_estimators': [100], 'max_depth': [10]}","[mfcc_3, wav_detail_lvl3_std, wav_detail_lvl4_...",0.917351,0.879518,0.958586,0.949900,0.908507,0.953933,0.867211,0.968291
2,RF,0.911406,0.915109,0.911378,0.911630,0.968960,"{'n_estimators': [100], 'max_depth': [10]}","[wav_detail_lvl5_std, mfcc_3, wav_detail_lvl3_...",0.915861,0.878479,0.956566,0.965849,0.906952,0.951740,0.866190,0.968960
3,RF,0.910911,0.914367,0.910878,0.911122,0.971069,"{'n_estimators': [100], 'max_depth': [10]}","[wav_detail_lvl5_std, mfcc_3, wav_detail_lvl3_...",0.915254,0.879070,0.954545,0.963261,0.906567,0.949664,0.867211,0.971069
4,RF,0.910902,0.914519,0.910873,0.911122,0.965302,"{'n_estimators': [100], 'max_depth': [10]}","[mfcc_3, wav_detail_lvl3_std, wav_detail_lvl4_...",0.915336,0.878366,0.955556,0.951504,0.906467,0.950673,0.866190,0.965302


🥇 Campeão Geral do Nível 1: RF


In [ ]:
# ==============================================================================
# 🏆 Busca Exaustiva (1023 Combinações) - Random Forest (RF)
# ==============================================================================
import itertools
import time
import pandas as pd
from IPython.display import display

print("="*60)
print("🔍 Iniciando Busca Combinatória Exaustiva - Random Forest (RF)")
print("="*60)

rf_config = ClassifierConfig(model_type='rf', random_state=42)
rf_grid = {'n_estimators': [100], 'max_depth': [None, 10]} 
tuning_config_rf = TuningConfig(param_grid=rf_grid, scoring_metric='f1_macro', n_jobs=-1)

combinatorial_results_rf = []
contador = 0
start_time = time.time()

for r in range(1, 11):
    for subset in itertools.combinations(top_10_features, r):
        contador += 1
        subset_list = list(subset)
        
        try:
            X_tr_sub = X_train_selected_metrics[subset_list]
            X_va_sub = X_val_selected_metrics[subset_list]
            
            rf_base = ClassifierFactory.build(rf_config)
            rf_champion = ModelOptimizer.optimize(
                estimator=rf_base, X_train=X_tr_sub, y_train=y_train_enc, 
                X_val=X_va_sub, y_val=y_val_enc, config=tuning_config_rf
            )
            
            # Executa a avaliação (certifique-se que return_report_dict=True no val_config)
            metrics = ModelEvaluator.evaluate(model=rf_champion, X_test=X_va_sub, y_test=y_val_enc, config=val_config)
            
            # Extração Robusta das Métricas
            f1    = metrics.get('f1_macro', 0.0)
            prauc = metrics.get('pr_auc_macro', metrics.get('pr_auc', 0.0))
            
            # Aceder ao dicionário interno do classification_report
            report = metrics.get('report', metrics)
            acc = report.get('accuracy', 0.0)
            rec = report.get('macro avg', {}).get('recall', 0.0)

            combinatorial_results_rf.append({
                'Tamanho': r, 
                'F1_Macro': f1,
                'Accuracy': acc,
                'Recall_Macro': rec,
                'PR_AUC': prauc,
                'Features': " + ".join(subset_list)
            })
        except Exception as e:
            continue
        
        if contador % 100 == 0:
            print(f"   -> Progresso RF: {contador}/1023... Melhor F1: {max([res['F1_Macro'] for res in combinatorial_results_rf]):.4f}")

df_comb_rf = pd.DataFrame(combinatorial_results_rf).sort_values(by='F1_Macro', ascending=False).reset_index(drop=True)
print(f"\n✅ Busca RF concluída!")
display(df_comb_rf.head(5))

🔍 Iniciando Busca Combinatória Exaustiva - Random Forest (RF)
12:37:55 - [ClassifierFactory] - INFO - CPU Scaling: Requested -1 -> Allocated 14 cores.
12:37:55 - [ClassifierFactory] - INFO - Building registered model architecture: RF
12:37:55 - [HyperparameterTuner] - INFO - Starting Optimization Engine. Target Metric: f1_macro


Fitting 1 folds for each of 2 candidates, totalling 2 fits
12:38:11 - [HyperparameterTuner] - INFO - Optimization complete. Champion Score (f1_macro) on Val Set: 0.3147
12:38:11 - [HyperparameterTuner] - INFO - Champion Hyperparameters: {'max_depth': None, 'n_estimators': 100}
12:38:11 - [HyperparameterTuner] - INFO - Refitting the champion model exclusively on the Training Set...
12:38:11 - [ModelEvaluator] - INFO - Executing Rigorous Stress Test (Model Evaluation)...
12:38:11 - [ModelEvaluator] - INFO - Test Results -> F1-Macro: 0.3147 | PR-AUC: 0.31114415475414303
12:38:11 - [ClassifierFactory] - INFO - Building registered model architecture: RF
12:38:11 - [HyperparameterTuner] - INFO - Starting Optimization Engine. Target Metric: f1_macro
Fitting 1 folds for each of 2 candidates, totalling 2 fits
12:38:15 - [HyperparameterTuner] - INFO - Optimization complete. Champion Score (f1_macro) on Val Set: 0.3149
12:38:15 - [HyperparameterTuner] - INFO - Champion Hyperparameters: {'max_dept

,Tamanho,F1_Macro,Accuracy,Recall_Macro,PR_AUC,Features
0,3,0.462954,0.0,0.0,0.447632,wav_detail_lvl5_std + mfcc_3 + mfcc_5
1,3,0.450734,0.0,0.0,0.439791,mfcc_3 + wav_detail_lvl4_std + mfcc_5
2,3,0.449913,0.0,0.0,0.439335,mfcc_3 + wav_detail_lvl3_std + mfcc_5
3,7,0.449628,0.0,0.0,0.432182,wav_detail_lvl5_std + mfcc_3 + wav_detail_lvl3...
4,4,0.449534,0.0,0.0,0.435718,wav_detail_lvl5_std + mfcc_3 + wav_detail_lvl3...


In [ ]:
# ==============================================================================
# 🏆 Busca Exaustiva (1023 Combinações) - XGBoost (XGB)
# ==============================================================================
print("="*60)
print("🔍 Iniciando Busca Combinatória Exaustiva - XGBoost (XGB)")
print("="*60)

xgb_config = ClassifierConfig(model_type='xgb', random_state=42)
xgb_grid = {'max_depth': [3, 5], 'learning_rate': [0.1]} 
tuning_config_xgb = TuningConfig(param_grid=xgb_grid, scoring_metric='f1_macro', n_jobs=-1)

combinatorial_results_xgb = []
contador = 0

for r in range(1, 11):
    for subset in itertools.combinations(top_10_features, r):
        contador += 1
        subset_list = list(subset)
        
        try:
            X_tr_sub = X_train_selected_metrics[subset_list]
            X_va_sub = X_val_selected_metrics[subset_list]
            
            xgb_base = ClassifierFactory.build(xgb_config)
            xgb_champion = ModelOptimizer.optimize(
                estimator=xgb_base, X_train=X_tr_sub, y_train=y_train_enc, 
                X_val=X_va_sub, y_val=y_val_enc, config=tuning_config_xgb
            )
            
            metrics = ModelEvaluator.evaluate(model=xgb_champion, X_test=X_va_sub, y_test=y_val_enc, config=val_config)
            
            f1    = metrics.get('f1_macro', 0.0)
            prauc = metrics.get('pr_auc_macro', metrics.get('pr_auc', 0.0))
            
            report = metrics.get('report', metrics)
            acc = report.get('accuracy', 0.0)
            rec = report.get('macro avg', {}).get('recall', 0.0)
            
            combinatorial_results_xgb.append({
                'Tamanho': r, 'F1_Macro': f1, 'Accuracy': acc, 'Recall_Macro': rec, 'PR_AUC': prauc,
                'Features': " + ".join(subset_list)
            })
        except:
            continue
            
        if contador % 100 == 0:
            print(f"   -> Progresso XGB: {contador}/1023... Melhor F1: {max([res['F1_Macro'] for res in combinatorial_results_xgb]):.4f}")

df_comb_xgb = pd.DataFrame(combinatorial_results_xgb).sort_values(by='F1_Macro', ascending=False).reset_index(drop=True)
print(f"\n✅ Busca XGB concluída!")
display(df_comb_xgb.head(5))

🔍 Iniciando Busca Combinatória Exaustiva - XGBoost (XGB)
13:17:55 - [ClassifierFactory] - INFO - CPU Scaling: Requested -1 -> Allocated 14 cores.
13:17:55 - [ClassifierFactory] - INFO - Building registered model architecture: XGB


13:17:56 - [HyperparameterTuner] - INFO - Starting Optimization Engine. Target Metric: f1_macro
Fitting 1 folds for each of 2 candidates, totalling 2 fits
13:17:59 - [HyperparameterTuner] - INFO - Optimization complete. Champion Score (f1_macro) on Val Set: 0.3002
13:17:59 - [HyperparameterTuner] - INFO - Champion Hyperparameters: {'learning_rate': 0.1, 'max_depth': 3}
13:17:59 - [HyperparameterTuner] - INFO - Refitting the champion model exclusively on the Training Set...
13:18:00 - [ModelEvaluator] - INFO - Executing Rigorous Stress Test (Model Evaluation)...
13:18:00 - [ModelEvaluator] - INFO - Test Results -> F1-Macro: 0.3002 | PR-AUC: 0.36632210829431505
13:18:00 - [ClassifierFactory] - INFO - Building registered model architecture: XGB
13:18:00 - [HyperparameterTuner] - INFO - Starting Optimization Engine. Target Metric: f1_macro
Fitting 1 folds for each of 2 candidates, totalling 2 fits
13:18:02 - [HyperparameterTuner] - INFO - Optimization complete. Champion Score (f1_macro) on

,Tamanho,F1_Macro,Accuracy,Recall_Macro,PR_AUC,Features
0,7,0.445396,0.0,0.0,0.428977,wav_detail_lvl5_std + mfcc_3 + wav_detail_lvl3...
1,5,0.444863,0.0,0.0,0.426824,mfcc_3 + wav_detail_lvl3_std + rms_mean + mae ...
2,5,0.444291,0.0,0.0,0.437135,wav_detail_lvl5_std + mfcc_3 + wav_detail_lvl3...
3,3,0.444128,0.0,0.0,0.434991,wav_detail_lvl5_std + mfcc_3 + mfcc_5
4,4,0.443834,0.0,0.0,0.442719,wav_detail_lvl5_std + mfcc_3 + rms_mean + mfcc_5


In [ ]:
# ==============================================================================
# 🏆 Busca Exaustiva (1023 Combinações) - Logistic Regression (LR)
# ==============================================================================
print("="*60)
print("🔍 Iniciando Busca Combinatória Exaustiva - Regressão Logística (LR)")
print("="*60)

lr_config = ClassifierConfig(model_type='lr', random_state=42)
lr_grid = {'C': [1.0], 'max_iter': [1000]} # Grid estável para convergência
tuning_config_lr = TuningConfig(param_grid=lr_grid, scoring_metric='f1_macro', n_jobs=-1)

combinatorial_results_lr = []
contador = 0
start_time = time.time()

for r in range(1, 11):
    for subset in itertools.combinations(top_10_features, r):
        contador += 1
        subset_list = list(subset)
        
        try:
            X_tr_sub = X_train_selected_metrics[subset_list]
            X_va_sub = X_val_selected_metrics[subset_list]
            
            lr_base = ClassifierFactory.build(lr_config)
            lr_champion = ModelOptimizer.optimize(
                estimator=lr_base, X_train=X_tr_sub, y_train=y_train_enc, 
                X_val=X_va_sub, y_val=y_val_enc, config=tuning_config_lr
            )
            
            metrics = ModelEvaluator.evaluate(model=lr_champion, X_test=X_va_sub, y_test=y_val_enc, config=val_config)
            
            # Extração Robusta (Indo buscar ao dicionário interno do report)
            f1    = metrics.get('f1_macro', 0.0)
            prauc = metrics.get('pr_auc_macro', metrics.get('pr_auc', 0.0))
            
            report = metrics.get('report', metrics)
            acc = report.get('accuracy', 0.0)
            rec = report.get('macro avg', {}).get('recall', 0.0)
            
            combinatorial_results_lr.append({
                'Tamanho': r, 
                'F1_Macro': f1, 
                'Accuracy': acc, 
                'Recall_Macro': rec, 
                'PR_AUC': prauc,
                'Features': " + ".join(subset_list)
            })
        except:
            continue

df_comb_lr = pd.DataFrame(combinatorial_results_lr).sort_values(by='F1_Macro', ascending=False).reset_index(drop=True)
print(f"✅ Busca LR concluída!")
display(df_comb_lr.head(5))

🔍 Iniciando Busca Combinatória Exaustiva - Regressão Logística (LR)
13:43:00 - [ClassifierFactory] - INFO - CPU Scaling: Requested -1 -> Allocated 14 cores.
13:43:00 - [ClassifierFactory] - INFO - Building registered model architecture: LR
13:43:00 - [HyperparameterTuner] - INFO - Starting Optimization Engine. Target Metric: f1_macro
Fitting 1 folds for each of 1 candidates, totalling 1 fits
13:43:00 - [HyperparameterTuner] - INFO - Optimization complete. Champion Score (f1_macro) on Val Set: 0.3368
13:43:00 - [HyperparameterTuner] - INFO - Champion Hyperparameters: {'C': 1.0, 'max_iter': 1000}
13:43:00 - [HyperparameterTuner] - INFO - Refitting the champion model exclusively on the Training Set...
13:43:00 - [ModelEvaluator] - INFO - Executing Rigorous Stress Test (Model Evaluation)...
13:43:00 - [ModelEvaluator] - INFO - Test Results -> F1-Macro: 0.3368 | PR-AUC: 0.3746323841396487
13:43:00 - [ClassifierFactory] - INFO - Building registered model architecture: LR
13:43:00 - [Hyperpar

,Tamanho,F1_Macro,Accuracy,Recall_Macro,PR_AUC,Features
0,4,0.452623,0.0,0.0,0.443937,mfcc_6 + mfcc_3 + wav_detail_lvl3_std + mfcc_std
1,5,0.450023,0.0,0.0,0.452951,mfcc_6 + mfcc_3 + mfcc_1 + mfcc_std + mfcc_5
2,5,0.448808,0.0,0.0,0.441535,mfcc_6 + mfcc_3 + wav_detail_lvl3_std + mfcc_1...
3,3,0.448701,0.0,0.0,0.441887,mfcc_6 + mfcc_1 + mfcc_std
4,4,0.448203,0.0,0.0,0.437044,mfcc_6 + mfcc_3 + mae + mfcc_5


In [ ]:
# ==============================================================================
#Busca Exaustiva (1023 Combinações) - SGD
# ==============================================================================
print("="*60)
print("🔍 Iniciando Busca Combinatória Exaustiva - SGD")
print("="*60)

sgd_config = ClassifierConfig(model_type='sgd', random_state=42)
sgd_grid = {'alpha': [0.001], 'penalty': ['l2']} 
tuning_config_sgd = TuningConfig(param_grid=sgd_grid, scoring_metric='f1_macro', n_jobs=-1)

combinatorial_results_sgd = []
contador = 0

for r in range(1, 11):
    for subset in itertools.combinations(top_10_features, r):
        contador += 1
        subset_list = list(subset)
        
        try:
            X_tr_sub = X_train_selected_metrics[subset_list]
            X_va_sub = X_val_selected_metrics[subset_list]
            
            sgd_base = ClassifierFactory.build(sgd_config)
            sgd_champion = ModelOptimizer.optimize(
                estimator=sgd_base, X_train=X_tr_sub, y_train=y_train_enc, 
                X_val=X_va_sub, y_val=y_val_enc, config=tuning_config_sgd
            )
            
            metrics = ModelEvaluator.evaluate(model=sgd_champion, X_test=X_va_sub, y_test=y_val_enc, config=val_config)
            
            # Extração Robusta
            f1    = metrics.get('f1_macro', 0.0)
            prauc = metrics.get('pr_auc_macro', metrics.get('pr_auc', 0.0))
            
            report = metrics.get('report', metrics)
            acc = report.get('accuracy', 0.0)
            rec = report.get('macro avg', {}).get('recall', 0.0)
            
            combinatorial_results_sgd.append({
                'Tamanho': r, 
                'F1_Macro': f1, 
                'Accuracy': acc, 
                'Recall_Macro': rec, 
                'PR_AUC': prauc,
                'Features': " + ".join(subset_list)
            })
        except:
            continue

df_comb_sgd = pd.DataFrame(combinatorial_results_sgd).sort_values(by='F1_Macro', ascending=False).reset_index(drop=True)
print(f"✅ Busca SGD concluída!")
display(df_comb_sgd.head(5))

🔍 Iniciando Busca Combinatória Exaustiva - SGD
13:51:28 - [ClassifierFactory] - INFO - CPU Scaling: Requested -1 -> Allocated 14 cores.
13:51:28 - [ClassifierFactory] - INFO - Building registered model architecture: SGD
13:51:28 - [HyperparameterTuner] - INFO - Starting Optimization Engine. Target Metric: f1_macro
Fitting 1 folds for each of 1 candidates, totalling 1 fits
13:51:28 - [HyperparameterTuner] - INFO - Optimization complete. Champion Score (f1_macro) on Val Set: 0.2817
13:51:28 - [HyperparameterTuner] - INFO - Champion Hyperparameters: {'alpha': 0.001, 'penalty': 'l2'}
13:51:28 - [HyperparameterTuner] - INFO - Refitting the champion model exclusively on the Training Set...
13:51:28 - [ModelEvaluator] - INFO - Executing Rigorous Stress Test (Model Evaluation)...
13:51:28 - [ModelEvaluator] - INFO - Test Results -> F1-Macro: 0.2817 | PR-AUC: 0.3953710554382414
13:51:28 - [ClassifierFactory] - INFO - Building registered model architecture: SGD
13:51:28 - [HyperparameterTuner] -

,Tamanho,F1_Macro,Accuracy,Recall_Macro,PR_AUC,Features
0,5,0.456923,0.0,0.0,0.464203,mfcc_6 + wav_detail_lvl5_std + mfcc_3 + wav_de...
1,5,0.456510,0.0,0.0,0.466606,mfcc_6 + wav_detail_lvl5_std + wav_detail_lvl3...
2,6,0.455974,0.0,0.0,0.465701,mfcc_6 + wav_detail_lvl5_std + wav_detail_lvl3...
3,5,0.455954,0.0,0.0,0.463959,mfcc_6 + wav_detail_lvl5_std + wav_detail_lvl3...
4,6,0.455848,0.0,0.0,0.464484,mfcc_6 + wav_detail_lvl5_std + wav_detail_lvl3...
